In [25]:
"""
EventEdge — Iron Condor Backtester
RBI Event Volatility Study

A delta-based Iron Condor strategy, backtested on NSE historical option
chain data across 10 RBI policy events and 2 real market shock days.

Core finding: a tight (0.20 delta) condor shows a perfect win rate on
calm, scheduled events but loses money net once real shocks are included.
A wider (0.10 delta) condor sacrifices per-trade income but survives
shocks well enough to be net profitable overall.
"""

print("EventEdge — Iron Condor Backtester loaded.")

EventEdge — Iron Condor Backtester loaded.


In [26]:
# CELL 1: Setup
!pip install requests pandas scipy -q

import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime, timedelta
import io
import zipfile
import time
import json

print("Setup complete.")

Setup complete.


In [27]:
# CELL 2: NSE Bhavcopy Downloader (historical data, no auth needed)

def download_fo_bhavcopy(date_str):
    """
    Downloads NSE F&O UDiFF Bhavcopy for a given date.
    Returns None if unavailable (holiday, future date, etc.) instead of raising.
    """
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    yyyymmdd = dt.strftime("%Y%m%d")

    url = f"https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_{yyyymmdd}_F_0000.csv.zip"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "/",
        "Referer": "https://www.nseindia.com/all-reports-derivatives"
    }

    try:
        session = requests.Session()
        session.get("https://www.nseindia.com", headers=headers, timeout=10)
        response = session.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            return None

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            csv_filename = z.namelist()[0]
            with z.open(csv_filename) as f:
                df = pd.read_csv(f)
        return df

    except Exception:
        return None

print("download_fo_bhavcopy defined.")

download_fo_bhavcopy defined.


In [28]:
# CELL 3: Clean Historical Nifty Option Chain

def get_historical_nifty_chain(date_str, expiry_filter=None):
    """
    Fetches NSE F&O bhavcopy for date_str (YYYY-MM-DD) and returns
    clean Nifty option chain data. Returns empty DataFrame if no data.
    """
    df_raw = download_fo_bhavcopy(date_str)

    if df_raw is None:
        return pd.DataFrame()

    df_opts = df_raw[(df_raw['TckrSymb'] == 'NIFTY') & (df_raw['FinInstrmTp'] == 'IDO')].copy()

    if df_opts.empty:
        return pd.DataFrame()

    cols = ['TradDt', 'XpryDt', 'StrkPric', 'OptnTp', 'OpnPric', 'HghPric',
            'LwPric', 'ClsPric', 'TtlTradgVol', 'OpnIntrst', 'ChngInOpnIntrst']
    df_opts = df_opts[cols].rename(columns={
        'StrkPric': 'strike', 'OptnTp': 'option_type', 'ClsPric': 'close_price',
        'OpnIntrst': 'oi', 'ChngInOpnIntrst': 'change_oi', 'TtlTradgVol': 'volume',
        'XpryDt': 'expiry', 'TradDt': 'date'
    })

    if expiry_filter:
        df_opts = df_opts[df_opts['expiry'] == expiry_filter]

    return df_opts.reset_index(drop=True)

print("get_historical_nifty_chain defined.")

get_historical_nifty_chain defined.


In [29]:
# CELL 4: Black-Scholes Pricing, Greeks, and IV Solver

RISK_FREE_RATE = 0.065

def bs_price(S, K, T, r, sigma, option_type):
    if T <= 0 or sigma <= 0:
        if option_type == "CE":
            return max(S - K, 0)
        else:
            return max(K - S, 0)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "CE":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price


def bs_greeks(S, K, T, r, sigma, option_type):
    if T <= 0 or sigma <= 0:
        return {"delta": 0, "gamma": 0, "theta": 0, "vega": 0}

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf_d1 = norm.pdf(d1)

    if option_type == "CE":
        delta = norm.cdf(d1)
        theta = (-S * pdf_d1 * sigma / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        delta = norm.cdf(d1) - 1
        theta = (-S * pdf_d1 * sigma / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365

    gamma = pdf_d1 / (S * sigma * np.sqrt(T))
    vega = S * pdf_d1 * np.sqrt(T) / 100

    return {"delta": delta, "gamma": gamma, "theta": theta, "vega": vega}


def implied_volatility(market_price, S, K, T, r, option_type):
    if T <= 0:
        return None

    def objective(sigma):
        return bs_price(S, K, T, r, sigma, option_type) - market_price

    try:
        iv = brentq(objective, 1e-6, 5.0, xtol=1e-6)
        return iv
    except ValueError:
        return None

print("Black-Scholes module defined.")

Black-Scholes module defined.


In [30]:
# CELL 5: Spot Price Estimator via Put-Call Parity

def estimate_spot_from_chain(df_chain):
    """
    Estimates underlying spot price using put-call parity:
    Spot ≈ Strike + (Call price - Put price), at the strike where
    Call and Put prices are closest (synthetic ATM point).
    """
    calls = df_chain[df_chain['option_type'] == 'CE'][['strike', 'close_price']].rename(columns={'close_price': 'call_price'})
    puts = df_chain[df_chain['option_type'] == 'PE'][['strike', 'close_price']].rename(columns={'close_price': 'put_price'})

    merged = pd.merge(calls, puts, on='strike')
    merged = merged[(merged['call_price'] > 0) & (merged['put_price'] > 0)]

    if merged.empty:
        return None

    merged['diff'] = (merged['call_price'] - merged['put_price']).abs()
    atm_row = merged.loc[merged['diff'].idxmin()]

    spot_estimate = atm_row['strike'] + (atm_row['call_price'] - atm_row['put_price'])
    return spot_estimate

print("estimate_spot_from_chain defined.")

estimate_spot_from_chain defined.


In [31]:
# CELL 6: Apply Black-Scholes IV + Greeks to a historical option chain DataFrame

def enrich_with_greeks(df_chain, spot_price, strike_range_pct=0.06):
    """
    Adds implied_vol, delta, gamma, theta, vega columns to an option chain.
    Filters to strikes within strike_range_pct of spot to avoid unstable IV
    solves on illiquid, deep ITM/OTM strikes.
    """
    df = df_chain.copy()

    lower_bound = spot_price * (1 - strike_range_pct)
    upper_bound = spot_price * (1 + strike_range_pct)
    df = df[(df['strike'] >= lower_bound) & (df['strike'] <= upper_bound)].copy()

    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
    df['expiry'] = pd.to_datetime(df['expiry'], format='%Y-%m-%d', errors='coerce')
    df['days_to_expiry'] = (df['expiry'] - df['date']).dt.days
    df['T'] = df['days_to_expiry'] / 365

    ivs, deltas, gammas, thetas, vegas = [], [], [], [], []

    for _, row in df.iterrows():
        S = spot_price
        K = row['strike']
        T = row['T']
        opt_type = row['option_type']
        market_price = row['close_price']

        if market_price <= 0 or T <= 0:
            ivs.append(None); deltas.append(None); gammas.append(None); thetas.append(None); vegas.append(None)
            continue

        iv = implied_volatility(market_price, S, K, T, RISK_FREE_RATE, opt_type)
        if iv is None:
            ivs.append(None); deltas.append(None); gammas.append(None); thetas.append(None); vegas.append(None)
            continue

        greeks = bs_greeks(S, K, T, RISK_FREE_RATE, iv, opt_type)
        ivs.append(iv)
        deltas.append(greeks['delta'])
        gammas.append(greeks['gamma'])
        thetas.append(greeks['theta'])
        vegas.append(greeks['vega'])

    df['implied_vol'] = ivs
    df['delta'] = deltas
    df['gamma'] = gammas
    df['theta'] = thetas
    df['vega'] = vegas

    return df

print("enrich_with_greeks defined.")

enrich_with_greeks defined.


In [32]:
# CELL 7: Iron Condor Strategy Logic

def select_iron_condor_strikes(df_enriched, target_short_delta=0.20, target_long_delta=0.10):
    """
    Selects 4 strikes for an Iron Condor: short call, short put,
    long call (hedge), long put (hedge), based on delta targets.
    """
    calls = df_enriched[df_enriched['option_type'] == 'CE'].dropna(subset=['delta']).copy()
    puts = df_enriched[df_enriched['option_type'] == 'PE'].dropna(subset=['delta']).copy()

    if calls.empty or puts.empty:
        return None

    calls['delta_diff'] = (calls['delta'] - target_short_delta).abs()
    short_call = calls.loc[calls['delta_diff'].idxmin()]

    puts['delta_diff'] = (puts['delta'] - (-target_short_delta)).abs()
    short_put = puts.loc[puts['delta_diff'].idxmin()]

    calls_otm_of_short = calls[calls['strike'] > short_call['strike']]
    if calls_otm_of_short.empty:
        return None
    calls_otm_of_short = calls_otm_of_short.copy()
    calls_otm_of_short['delta_diff'] = (calls_otm_of_short['delta'] - target_long_delta).abs()
    long_call = calls_otm_of_short.loc[calls_otm_of_short['delta_diff'].idxmin()]

    puts_otm_of_short = puts[puts['strike'] < short_put['strike']]
    if puts_otm_of_short.empty:
        return None
    puts_otm_of_short = puts_otm_of_short.copy()
    puts_otm_of_short['delta_diff'] = (puts_otm_of_short['delta'] - (-target_long_delta)).abs()
    long_put = puts_otm_of_short.loc[puts_otm_of_short['delta_diff'].idxmin()]

    credit_received = (short_call['close_price'] + short_put['close_price']
                        - long_call['close_price'] - long_put['close_price'])

    return {
        "short_call_strike": short_call['strike'], "short_call_premium": short_call['close_price'], "short_call_delta": short_call['delta'],
        "short_put_strike": short_put['strike'], "short_put_premium": short_put['close_price'], "short_put_delta": short_put['delta'],
        "long_call_strike": long_call['strike'], "long_call_premium": long_call['close_price'], "long_call_delta": long_call['delta'],
        "long_put_strike": long_put['strike'], "long_put_premium": long_put['close_price'], "long_put_delta": long_put['delta'],
        "credit_received": credit_received
    }

print("select_iron_condor_strikes defined.")

select_iron_condor_strikes defined.


In [33]:
# CELL 8: Multi-day Position Simulator — Helper Functions

def get_trading_days_between(start_date, end_date):
    """Returns list of weekday dates between start and end (inclusive)."""
    days = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    while current <= end:
        if current.weekday() < 5:
            days.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)
    return days


def get_condor_value_on_date(date_str, expiry_date, condor_strikes):
    """Returns cost to close the condor on date_str, or None if data missing."""
    df_day = get_historical_nifty_chain(date_str, expiry_filter=expiry_date)

    if df_day.empty:
        return None

    def get_leg_price(strike, opt_type):
        match = df_day[(df_day['strike'] == strike) & (df_day['option_type'] == opt_type)]
        if match.empty:
            return None
        return match.iloc[0]['close_price']

    sc_price = get_leg_price(condor_strikes['short_call_strike'], 'CE')
    sp_price = get_leg_price(condor_strikes['short_put_strike'], 'PE')
    lc_price = get_leg_price(condor_strikes['long_call_strike'], 'CE')
    lp_price = get_leg_price(condor_strikes['long_put_strike'], 'PE')

    if None in [sc_price, sp_price, lc_price, lp_price]:
        return None

    cost_to_close = (sc_price + sp_price) - (lc_price + lp_price)
    return cost_to_close


def find_nearest_expiry(df_raw, after_date_str, min_days_ahead=1):
    """Finds nearest expiry strictly after after_date_str by at least min_days_ahead days."""
    if df_raw.empty:
        return None
    expiries = sorted(df_raw['expiry'].unique())
    after_date = datetime.strptime(after_date_str, "%Y-%m-%d") + timedelta(days=min_days_ahead)
    for exp in expiries:
        exp_date = datetime.strptime(exp, "%Y-%m-%d") if isinstance(exp, str) else exp
        if exp_date >= after_date:
            return exp
    return expiries[-1] if expiries else None

print("Simulator helper functions defined.")

Simulator helper functions defined.


In [34]:
# CELL 9: Full Iron Condor Simulator (configurable delta targets)

def simulate_iron_condor(entry_date, expiry_date=None, target_pct=0.50, sl_multiple=2.0,
                          target_short_delta=0.20, target_long_delta=0.10, strike_range_pct=0.12):
    """
    Full simulation: enters condor on entry_date, checks daily until
    target/stop-loss/expiry. Returns dict with entry/exit details and P&L.
    """
    df_entry_raw = get_historical_nifty_chain(entry_date)

    if df_entry_raw.empty:
        return {"error": f"No data on entry date {entry_date}"}

    if expiry_date is None:
        expiry_date = find_nearest_expiry(df_entry_raw, entry_date)
        if expiry_date is None:
            return {"error": f"Could not determine expiry from entry date {entry_date}"}

    df_entry_filtered = df_entry_raw[df_entry_raw['expiry'] == expiry_date].copy()
    if df_entry_filtered.empty:
        return {"error": f"No option data for expiry {expiry_date} on entry date {entry_date}"}

    spot_estimate = estimate_spot_from_chain(df_entry_filtered)
    if spot_estimate is None:
        return {"error": f"Could not estimate spot price on {entry_date}"}

    df_enriched_entry = enrich_with_greeks(df_entry_filtered, spot_price=spot_estimate, strike_range_pct=strike_range_pct)
    condor = select_iron_condor_strikes(df_enriched_entry, target_short_delta=target_short_delta, target_long_delta=target_long_delta)

    if condor is None:
        return {"error": f"Could not select condor strikes on {entry_date} for expiry {expiry_date}"}

    credit_received = condor['credit_received']
    target_value = credit_received * (1 - target_pct)
    sl_value = credit_received * sl_multiple

    trading_days = get_trading_days_between(entry_date, expiry_date)

    result = {
        "entry_date": entry_date, "expiry_date": expiry_date,
        "spot_estimate": spot_estimate, "condor": condor, "credit_received": credit_received,
        "target_value": target_value, "sl_value": sl_value,
        "exit_date": None, "exit_reason": None, "exit_value": None, "pnl": None
    }

    for day in trading_days[1:]:
        cost_to_close = get_condor_value_on_date(day, expiry_date, condor)

        if cost_to_close is None:
            continue

        if cost_to_close <= target_value:
            result.update({"exit_date": day, "exit_reason": "target", "exit_value": cost_to_close,
                            "pnl": credit_received - cost_to_close})
            return result

        if cost_to_close >= sl_value:
            result.update({"exit_date": day, "exit_reason": "stop_loss", "exit_value": cost_to_close,
                            "pnl": credit_received - cost_to_close})
            return result

    final_day = trading_days[-1]
    final_cost = get_condor_value_on_date(final_day, expiry_date, condor)
    result.update({"exit_date": final_day, "exit_reason": "expiry",
                    "exit_value": final_cost, "pnl": credit_received - (final_cost or 0)})
    return result

print("simulate_iron_condor defined.")

# Quick sanity test
test_result = simulate_iron_condor(entry_date="2025-04-07")
print(f"\nSanity test (2025-04-07): exit={test_result.get('exit_reason')}, pnl={test_result.get('pnl')}")

simulate_iron_condor defined.

Sanity test (2025-04-07): exit=target, pnl=105.35


In [35]:
# CELL 10: Full validation run — 10 RBI events + 2 shock events, 0.20 delta

def entry_date_before(event_date_str, days_before=2):
    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    entry = event_date - timedelta(days=days_before)
    while entry.weekday() >= 5:
        entry -= timedelta(days=1)
    return entry.strftime("%Y-%m-%d")


all_event_dates = [
    ('2024-08-08', 'RBI'), ('2024-10-09', 'RBI'), ('2024-12-06', 'RBI'), ('2025-02-07', 'RBI'), ('2025-04-09', 'RBI'),
    ('2025-06-06', 'RBI'), ('2025-08-06', 'RBI'), ('2025-10-01', 'RBI'), ('2025-12-05', 'RBI'), ('2026-02-06', 'RBI'),
    ('2024-06-04', 'SHOCK'), ('2026-03-19', 'SHOCK'),
]

results_020 = []
for event_date, tag in all_event_dates:
    entry = entry_date_before(event_date, days_before=2) if tag == 'RBI' else entry_date_before(event_date, days_before=1)
    result = simulate_iron_condor(entry_date=entry, target_short_delta=0.20, target_long_delta=0.10)
    result['event_date'] = event_date
    result['tag'] = tag
    results_020.append(result)

    if "error" in result:
        print(f"  [{tag}] {event_date} (entry {entry}): SKIPPED - {result['error']}")
    else:
        print(f"  [{tag}] {event_date} (entry {entry}): credit={result['credit_received']:.2f}, exit={result['exit_reason']}, pnl={result['pnl']:.2f}")
    time.sleep(1)

valid_020 = [r for r in results_020 if "error" not in r]
total_pnl_020 = sum(r['pnl'] for r in valid_020)
wins_020 = [r for r in valid_020 if r['pnl'] > 0]
print(f"\n=== 0.20 DELTA SUMMARY ===")
print(f"Valid trades: {len(valid_020)}/{len(all_event_dates)}")
print(f"Win rate: {len(wins_020)/len(valid_020)*100:.1f}%")
print(f"Total P&L: {total_pnl_020:.2f} | Avg: {total_pnl_020/len(valid_020):.2f} | Worst: {min(r['pnl'] for r in valid_020):.2f}")

  [RBI] 2024-08-08 (entry 2024-08-06): credit=49.20, exit=target, pnl=49.10
  [RBI] 2024-10-09 (entry 2024-10-07): credit=62.10, exit=target, pnl=46.05
  [RBI] 2024-12-06 (entry 2024-12-04): credit=39.95, exit=target, pnl=24.05
  [RBI] 2025-02-07 (entry 2025-02-05): credit=31.60, exit=target, pnl=31.35
  [RBI] 2025-04-09 (entry 2025-04-07): credit=105.40, exit=target, pnl=105.35
  [RBI] 2025-06-06 (entry 2025-06-04): credit=22.40, exit=target, pnl=22.25
  [RBI] 2025-08-06 (entry 2025-08-04): credit=32.95, exit=target, pnl=32.80
  [RBI] 2025-10-01 (entry 2025-09-29): credit=21.30, exit=target, pnl=21.10
  [RBI] 2025-12-05 (entry 2025-12-03): credit=36.05, exit=target, pnl=29.20
  [RBI] 2026-02-06 (entry 2026-02-04): credit=50.80, exit=target, pnl=27.75
  [SHOCK] 2024-06-04 (entry 2024-06-03): credit=105.25, exit=stop_loss, pnl=-317.10
  [SHOCK] 2026-03-19 (entry 2026-03-18): credit=73.45, exit=stop_loss, pnl=-141.20

=== 0.20 DELTA SUMMARY ===
Valid trades: 12/12
Win rate: 83.3%
Total P

In [36]:
# CELL 11: Same 12 events, 0.10 delta (wider/more conservative)

results_010 = []
for event_date, tag in all_event_dates:
    entry = entry_date_before(event_date, days_before=2) if tag == 'RBI' else entry_date_before(event_date, days_before=1)
    result = simulate_iron_condor(entry_date=entry, target_short_delta=0.10, target_long_delta=0.05)
    result['event_date'] = event_date
    result['tag'] = tag
    results_010.append(result)

    if "error" in result:
        print(f"  [{tag}] {event_date} (entry {entry}): SKIPPED - {result['error']}")
    else:
        print(f"  [{tag}] {event_date} (entry {entry}): credit={result['credit_received']:.2f}, exit={result['exit_reason']}, pnl={result['pnl']:.2f}")
    time.sleep(1)

valid_010 = [r for r in results_010 if "error" not in r]
total_pnl_010 = sum(r['pnl'] for r in valid_010)
wins_010 = [r for r in valid_010 if r['pnl'] > 0]
print(f"\n=== 0.10 DELTA SUMMARY ===")
print(f"Valid trades: {len(valid_010)}/{len(all_event_dates)}")
print(f"Win rate: {len(wins_010)/len(valid_010)*100:.1f}%")
print(f"Total P&L: {total_pnl_010:.2f} | Avg: {total_pnl_010/len(valid_010):.2f} | Worst: {min(r['pnl'] for r in valid_010):.2f}")

  [RBI] 2024-08-08 (entry 2024-08-06): credit=20.15, exit=target, pnl=20.10
  [RBI] 2024-10-09 (entry 2024-10-07): credit=25.90, exit=target, pnl=22.35
  [RBI] 2024-12-06 (entry 2024-12-04): credit=15.55, exit=target, pnl=14.40
  [RBI] 2025-02-07 (entry 2025-02-05): credit=10.05, exit=target, pnl=10.00
  [RBI] 2025-04-09 (entry 2025-04-07): credit=25.60, exit=target, pnl=25.60
  [RBI] 2025-06-06 (entry 2025-06-04): credit=14.90, exit=target, pnl=14.80
  [RBI] 2025-08-06 (entry 2025-08-04): credit=14.95, exit=target, pnl=7.70
  [RBI] 2025-10-01 (entry 2025-09-29): credit=8.80, exit=target, pnl=8.75
  [RBI] 2025-12-05 (entry 2025-12-03): credit=15.45, exit=target, pnl=8.70
  [RBI] 2026-02-06 (entry 2026-02-04): credit=23.40, exit=target, pnl=13.90
  [SHOCK] 2024-06-04 (entry 2024-06-03): SKIPPED - Could not select condor strikes on 2024-06-03 for expiry 2024-06-06
  [SHOCK] 2026-03-19 (entry 2026-03-18): credit=27.65, exit=stop_loss, pnl=-73.55

=== 0.10 DELTA SUMMARY ===
Valid trades: 1